# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kuteesatendojeremiah/Tendojerry-Flyrank/blob/main/work/notebooks/w06_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
%pip install -q duckdb huggingface_hub

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Same window as ML-04/ML-05/ML-06 (w03_data_contract.ipynb, w04_feature_leakage_check.ipynb,
# w05_signal_audit.ipynb) — mid-panel month, never the sealed June 2026 sample.
MONTH_START = "2026-03-01"
MONTH_END_EXCL = "2026-04-01"      # half-open: report_date < MONTH_END_EXCL
PREV30_START = "2026-01-30"        # the 30 days immediately before MONTH_START
PREV30_END_EXCL = MONTH_START

print(f"Connected. Iterating on month={MONTH_START[:7]} | prev30 window: [{PREV30_START}, {PREV30_END_EXCL})")


Paste your Hugging Face READ token (hf_...): ··········
Connected. Iterating on month=2026-03 | prev30 window: [2026-01-30, 2026-03-01)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Two signals checked first** (at least one must sit behind a real FlyRank flag):
1. **CTR-vs-position** (behind the CTR-fix logic / `needs_ctr_fix`) — claim: pages with a real
   CTR gap (CTR below what their position tier typically gets) are more likely declining.
2. **Staleness** (behind the refresh flags — this is literally my lane's own theme, Refresh /
   Content Opportunity Scoring) — claim: pages that haven't been updated in a long time are more
   likely declining.

Bucket tables + verdicts for both are in the code below.

**The rule:** a page is worth a refresh if it's visible (enough prev30 impressions to trust a
CTR reading), has a real CTR gap for its position tier, and gets weighted up the longer it's
gone since its last update — combining both checked signals into one transparent score, no
fitted weights (`skills/building-baselines/SKILL.md`).

**Score:** `ctr_gap × imp_prev30 × (1 + days_since_update / 365)`, computed only for visible
pages with a real position tier; 0 otherwise.

**Reason code (one):** `low_ctr_for_stale_position` when score > 0, else none.
**Action label:** `refresh` when score > 0, else `monitor`.

In [9]:
feature_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev30,
           SUM(gsc_clicks) AS clk_prev30,
           AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_prev30,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_prev30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{PREV30_START}' AND report_date < DATE '{PREV30_END_EXCL}'
    GROUP BY 1, 2
""").df()
feature_frame["ctr_prev30"] = (feature_frame["clk_prev30"] / feature_frame["imp_prev30"]).fillna(0)

content_dates = con.sql(f"""
    SELECT content_hash_id, content_updated_date
    FROM {TABLES['dim_content']}
""").df()
feature_frame = feature_frame.merge(content_dates, on="content_hash_id", how="left")

# dim_content is a CURRENT-state dimension table (as of whenever the warehouse was snapshotted),
# not a value frozen at MONTH_START — some rows carry an update dated AFTER MONTH_START, which
# would leak future information into "days since update" if used as-is. Only trust it when the
# update happened on or before MONTH_START; anything later (or missing) is treated as unknown,
# not a negative number.
update_date = pd.to_datetime(feature_frame["content_updated_date"])
month_start_ts = pd.Timestamp(MONTH_START)
valid_update = update_date <= month_start_ts
feature_frame["days_since_update"] = np.where(valid_update, (month_start_ts - update_date).dt.days, np.nan)

n_future_dated = int((~valid_update & update_date.notna()).sum())
print(f"{n_future_dated:,} items have a content_updated_date after MONTH_START — treated as unknown, not used as staleness.")

# --- Label (identical formula to ML-04/05/06) ---
march = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_march
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END_EXCL}'
    GROUP BY 1, 2
""").df()
data = feature_frame.merge(march, on=["client_hash_id", "content_hash_id"], how="inner")
data = data[data["imp_prev30"] > 0].copy()
data["is_declining"] = (data["imp_march"] < 0.8 * data["imp_prev30"]).astype(int)
print(f"{len(data):,} content items, base rate {data['is_declining'].mean():.3f}")

SIZE_FLOOR = 50  # skills/auditing-signals/SKILL.md: no verdict from a bucket under ~50 rows

def bucket_table(df, group_col, label_col="is_declining"):
    # observed=False so every category (even an empty one, e.g. "365+") still shows up in the
    # table with n=0 rather than silently disappearing and KeyError-ing call_verdict below.
    t = df.groupby(group_col, observed=False)[label_col].agg(n="count", decline_rate="mean")
    t["insufficient"] = t["n"] < SIZE_FLOOR
    return t

def call_verdict(t, best_bucket, worst_bucket, expected_direction, margin=0.03):
    if t.loc[best_bucket, "insufficient"] or t.loc[worst_bucket, "insufficient"]:
        return "insufficient data (a compared bucket is below the 50-row floor)"
    diff = t.loc[worst_bucket, "decline_rate"] - t.loc[best_bucket, "decline_rate"]
    signed_diff = diff if expected_direction == "up" else -diff
    if signed_diff >= margin:
        return "CONFIRMED"
    if signed_diff <= -margin:
        return "OPPOSITE"
    return "MIXED"

# --- Signal check 1: CTR-vs-position (behind needs_ctr_fix) ---
data["position_bucket"] = pd.cut(
    data["avg_position_prev30"], bins=[0, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)
ctr_by_position = data.dropna(subset=["position_bucket"]).groupby("position_bucket", observed=True).agg(
    total_clicks=("clk_prev30", "sum"), total_impressions=("imp_prev30", "sum")
)
ctr_by_position["expected_ctr"] = ctr_by_position["total_clicks"] / ctr_by_position["total_impressions"]
data["expected_ctr"] = data["position_bucket"].map(ctr_by_position["expected_ctr"]).astype(float)
data["has_ctr_gap"] = (data["expected_ctr"] - data["ctr_prev30"]) > 0

t_ctr = bucket_table(data.dropna(subset=["position_bucket"]), "has_ctr_gap")
print("\nSignal 1 — CTR gap for position (behind needs_ctr_fix) vs is_declining")
print(t_ctr)
v_ctr = call_verdict(t_ctr, best_bucket=False, worst_bucket=True, expected_direction="up")
print(f"Verdict: {v_ctr}")

# --- Signal check 2: staleness (behind the refresh flags) ---
data["staleness_bucket"] = pd.cut(
    data["days_since_update"], bins=[-1, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
)
data["staleness_bucket"] = data["staleness_bucket"].cat.add_categories(["unknown"]).fillna("unknown")
t_stale = bucket_table(data, "staleness_bucket")
print("\nSignal 2 — staleness (days since last update) vs is_declining")
print(t_stale)
v_stale = call_verdict(t_stale, best_bucket="0-30", worst_bucket="365+", expected_direction="up")
print(f"Verdict: {v_stale}")

# --- Encode the rule: ONE score, ONE reason code, ONE action label ---
VISIBILITY_FLOOR = 100  # below this, a CTR reading is too noisy to trust
visible = data["imp_prev30"] >= VISIBILITY_FLOOR
has_position = data["position_bucket"].notna()

ctr_gap = (data["expected_ctr"] - data["ctr_prev30"]).clip(lower=0)
staleness_multiplier = 1 + data["days_since_update"].fillna(0) / 365  # unknown -> neutral (1x), never negative now
data["score"] = np.where(visible & has_position, ctr_gap * data["imp_prev30"] * staleness_multiplier, 0.0)

data["reason_code"] = np.where(data["score"] > 0, "low_ctr_for_stale_position", None)
data["action"] = np.where(data["score"] > 0, "refresh", "monitor")

print(f"\nAction counts: {data['action'].value_counts().to_dict()}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

268,203 items have a content_updated_date after MONTH_START — treated as unknown, not used as staleness.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

146,253 content items, base rate 0.285

Signal 1 — CTR gap for position (behind needs_ctr_fix) vs is_declining
                  n  decline_rate  insufficient
has_ctr_gap                                    
False         28196      0.182473         False
True         116582      0.307655         False
Verdict: CONFIRMED

Signal 2 — staleness (days since last update) vs is_declining
                       n  decline_rate  insufficient
staleness_bucket                                    
0-30               28047      0.344921         False
31-90                353      0.297450         False
91-180              1128      0.531915         False
181-365              161      0.571429         False
365+                   0           NaN          True
unknown           116564      0.268265         False
Verdict: insufficient data (a compared bucket is below the 50-row floor)

Action counts: {'monitor': 91842, 'refresh': 54411}


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Ranks every evaluable item by `score` descending, writes the full queue to
`work/outputs/baseline_action_score.csv` (gitignored — regenerates each run, never committed).
Precision@50 is reported two ways: over this full development slice, and on the exact same
client-grouped held-out test clients ML-05's diagnostic used (same `GroupShuffleSplit` params,
so it's the identical client partition) — that second number is the one ML-08's model needs to
beat on equal footing.

In [10]:
ranked_queue = data.sort_values("score", ascending=False).reset_index(drop=True)
ranked_queue.insert(0, "rank", ranked_queue.index + 1)

os.makedirs("work/outputs", exist_ok=True)
output_cols = ["rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action",
               "position_bucket", "ctr_prev30", "expected_ctr", "days_since_update",
               "imp_prev30", "is_declining"]
ranked_queue[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(ranked_queue):,} rows to work/outputs/baseline_action_score.csv")

def precision_at_k(df, k, score_col="score", label_col="is_declining"):
    top = df.sort_values(score_col, ascending=False).head(k)
    return top[label_col].mean()

base_rate_full = data["is_declining"].mean()
p50_full = precision_at_k(data, 50)
print(f"\nFull-slice Precision@50: {p50_full:.3f} (base rate {base_rate_full:.3f}, n={len(data):,})")

# Same client-grouped test split as ML-05's diagnostic (identical GroupShuffleSplit params ->
# identical client partition, since it depends only on the set of client_hash_id groups).
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
_, test_idx = next(gss.split(data, data["is_declining"], groups=data["client_hash_id"]))
test_data = data.iloc[test_idx]

base_rate_test = test_data["is_declining"].mean()
p50_test = precision_at_k(test_data, 50)
print(f"Held-out (client-grouped) Precision@50: {p50_test:.3f} (base rate {base_rate_test:.3f}, n={len(test_data):,})")
if base_rate_test > 0:
    print(f"Lift over held-out base rate: {p50_test / base_rate_test:.2f}x")


Wrote 146,253 rows to work/outputs/baseline_action_score.csv

Full-slice Precision@50: 0.560 (base rate 0.285, n=146,253)
Held-out (client-grouped) Precision@50: 0.540 (base rate 0.320, n=75,733)
Lift over held-out base rate: 1.69x


## 3. Top-10 review

*For each of your top ten, one line each — the action, why it's there, and what would make it wrong.*

Auto-generated from the actual top-10 rows, not invented — each line reads the row's own
`ctr_prev30` / `expected_ctr` / `days_since_update` to state why it scored, and names the one
concrete caveat the data itself supports (a thin position-tier baseline, a very recent update,
or — for the rest — no obvious flag beyond the standing "title/meta may have already changed"
risk).

In [11]:
top10 = ranked_queue.head(10).copy()

def why_here(row):
    staleness_str = f"{int(row['days_since_update'])} days since last update" if pd.notna(row["days_since_update"]) else "no update date on record"
    return (f"CTR {row['ctr_prev30']*100:.2f}% vs expected {row['expected_ctr']*100:.2f}% for "
            f"'{row['position_bucket']}', {staleness_str}, "
            f"{int(row['imp_prev30']):,} prev30 impressions")

def what_would_make_it_wrong(row):
    if row["position_bucket"] == "deep":
        return "expected CTR for 'deep' positions is already tiny (0.07% per ML-06) — this gap is small in absolute terms even though it ranks high"
    if pd.notna(row["days_since_update"]) and row["days_since_update"] < 30:
        return "recently updated already — a CTR gap here may just need more time to show up, not another refresh"
    return "no obvious data-quality flag on this row — main risk is a recent title/meta change not yet reflected in this window"

for _, row in top10.iterrows():
    print(f"#{row['rank']}: action={row['action']} | why: {why_here(row)} | "
          f"what would make it wrong: {what_would_make_it_wrong(row)}")

top10[["rank", "action", "reason_code", "score", "position_bucket", "ctr_prev30", "expected_ctr", "days_since_update"]]


#1: action=refresh | why: CTR 0.00% vs expected 0.33% for 'page_1', no update date on record, 204,176 prev30 impressions | what would make it wrong: no obvious data-quality flag on this row — main risk is a recent title/meta change not yet reflected in this window
#2: action=refresh | why: CTR 0.00% vs expected 0.33% for 'page_1', no update date on record, 198,339 prev30 impressions | what would make it wrong: no obvious data-quality flag on this row — main risk is a recent title/meta change not yet reflected in this window
#3: action=refresh | why: CTR 0.00% vs expected 0.33% for 'page_1', 4 days since last update, 195,655 prev30 impressions | what would make it wrong: recently updated already — a CTR gap here may just need more time to show up, not another refresh
#4: action=refresh | why: CTR 0.00% vs expected 0.33% for 'page_1', no update date on record, 125,079 prev30 impressions | what would make it wrong: no obvious data-quality flag on this row — main risk is a recent title/met

,rank,action,reason_code,score,position_bucket,ctr_prev30,expected_ctr,days_since_update
0,1,refresh,low_ctr_for_stale_position,681.824404,page_1,0.000010,0.003349,NaN
1,2,refresh,low_ctr_for_stale_position,664.275177,page_1,0.000000,0.003349,NaN
2,3,refresh,low_ctr_for_stale_position,661.456206,page_1,0.000005,0.003349,4.0
3,4,refresh,low_ctr_for_stale_position,418.913451,page_1,0.000000,0.003349,NaN
4,5,refresh,low_ctr_for_stale_position,312.469386,top_3,0.000614,0.004170,NaN
5,6,refresh,low_ctr_for_stale_position,305.372056,top_3,0.002445,0.004170,NaN
6,7,refresh,low_ctr_for_stale_position,303.617895,page_1,0.000054,0.003349,NaN
7,8,refresh,low_ctr_for_stale_position,298.264090,page_1,0.000878,0.003349,NaN
8,9,refresh,low_ctr_for_stale_position,288.508971,page_1,0.000155,0.003349,NaN
9,10,refresh,low_ctr_for_stale_position,285.646262,page_1,0.001661,0.003349,NaN


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: any top-10 item in the `deep` position tier rides a tiny expected-CTR baseline
(0.07%, from ML-06), so its gap is numerically real but practically small next to a
`page_1`/`striking` pick with a similar score.

Leakage check: the rule reads only `imp_prev30` / `clk_prev30` / `avg_position_prev30`
(prev30-only, from `fact_daily`) and `content_updated_date`. That last one needed a fix —
`dim_content` holds each item's *current* state (as of whenever the warehouse was snapshotted),
not a value frozen at `MONTH_START`, so a page updated after March would otherwise leak a
future-dated "days since update" into the score. Section 1 now only trusts
`content_updated_date` when it falls on or before `MONTH_START`; anything later (or missing) is
treated as unknown, not a negative number. No March data, no product flags, and no future
window enters the score. `is_declining` is used only for evaluation in Sections 2/3, never as a
rule input.

In [12]:
weak_picks = top10[top10["position_bucket"] == "deep"]
print(f"Weak picks in the top 10: {len(weak_picks)} from the 'deep' position tier.")
if len(weak_picks):
    print(weak_picks[["rank", "score", "position_bucket", "ctr_prev30", "expected_ctr"]])

RULE_INPUTS = ["imp_prev30", "clk_prev30", "avg_position_prev30", "content_updated_date"]
print(f"\nRule inputs (all prev30 or static-metadata only): {RULE_INPUTS}")
print(f"prev30 window used: [{PREV30_START}, {PREV30_END_EXCL}) — none touch report_date >= MONTH_START.")
print(f"content_updated_date is only trusted when it falls on/before MONTH_START ({n_future_dated:,} future-dated rows treated as unknown, not used as staleness).")

# Same broader re-scan family as ML-05's Leakage test 3, applied to this notebook's rule inputs.
FLAG_LIKE = ("score", "flag", "priority", "risk", "declin", "trend", "recommend", "status")
schema_content = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()
hits = [c for c in schema_content["column_name"] if any(bad in c.lower() for bad in FLAG_LIKE) and c not in RULE_INPUTS]
print(f"dim_content flag-like columns NOT used as rule inputs: {hits or 'none found'}")


Weak picks in the top 10: 0 from the 'deep' position tier.

Rule inputs (all prev30 or static-metadata only): ['imp_prev30', 'clk_prev30', 'avg_position_prev30', 'content_updated_date']
prev30 window used: [2026-01-30, 2026-03-01) — none touch report_date >= MONTH_START.
content_updated_date is only trusted when it falls on/before MONTH_START (268,203 future-dated rows treated as unknown, not used as staleness).
dim_content flag-like columns NOT used as rule inputs: none found


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.